In [1]:
!pip install -q groq requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.2 MB/s eta 0:00:00


In [2]:
import os
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"

In [3]:
%%writefile state.py

from dataclasses import dataclass, field
from typing import Optional


class Stage:
    GREET           = "greet"
    LOOKUP          = "lookup"
    VERIFY          = "verify"
    SHOW_BALANCE    = "show_balance"
    COLLECT_CARD    = "collect_card"
    PROCESS_PAYMENT = "process_payment"
    DONE            = "done"


@dataclass
class AccountData:
    account_id: str
    full_name: str
    dob: str
    aadhaar_last4: str
    pincode: str
    balance: float


@dataclass
class CardData:
    cardholder_name: Optional[str] = None
    card_number: Optional[str] = None
    cvv: Optional[str] = None
    expiry_month: Optional[int] = None
    expiry_year: Optional[int] = None


@dataclass
class ConversationState:
    stage: str = Stage.GREET

    account_id: Optional[str] = None
    account_data: Optional[AccountData] = None

    provided_name: Optional[str] = None
    provided_dob: Optional[str] = None
    provided_aadhaar: Optional[str] = None
    provided_pincode: Optional[str] = None
    is_verified: bool = False

    verify_attempts: int = 0
    secondary_attempts: int = 0
    MAX_VERIFY_ATTEMPTS: int = 3
    MAX_SECONDARY_ATTEMPTS: int = 3

    payment_amount: Optional[float] = None
    card_data: CardData = field(default_factory=CardData)
    transaction_id: Optional[str] = None
    payment_error: Optional[str] = None

Writing state.py


In [4]:
%%writefile tools.py

import requests
from state import AccountData

BASE_URL = "https://se-payment-verification-api.service.external.usea2.aws.prodigaltech.com"


def lookup_account(account_id: str):
    try:
        response = requests.post(
            f"{BASE_URL}/api/lookup-account",
            json={"account_id": account_id},
            timeout=10
        )
        if response.status_code == 200:
            data = response.json()
            return AccountData(
                account_id=data["account_id"],
                full_name=data["full_name"],
                dob=data["dob"],
                aadhaar_last4=data["aadhaar_last4"],
                pincode=data["pincode"],
                balance=data["balance"]
            )
        elif response.status_code == 404:
            return None
        else:
            raise Exception(f"Unexpected API error: {response.status_code}")
    except requests.exceptions.Timeout:
        raise Exception("The server took too long to respond. Please try again.")
    except requests.exceptions.ConnectionError:
        raise Exception("Could not connect to the server. Please try again.")


def process_payment(account_id: str, amount: float, card: dict):
    payload = {
        "account_id": account_id,
        "amount": amount,
        "payment_method": {
            "type": "card",
            "card": card
        }
    }
    try:
        response = requests.post(
            f"{BASE_URL}/api/process-payment",
            json=payload,
            timeout=10
        )
        data = response.json()
        if response.status_code == 200 and data.get("success"):
            return {"success": True, "transaction_id": data["transaction_id"]}
        else:
            return {"success": False, "error_code": data.get("error_code", "unknown_error")}
    except requests.exceptions.Timeout:
        raise Exception("Payment server timed out. Please try again.")
    except requests.exceptions.ConnectionError:
        raise Exception("Could not connect to payment server. Please try again.")

Writing tools.py


In [5]:
%%writefile verifier.py

from state import AccountData


def verify_identity(account: AccountData, provided_name: str,
                    provided_dob: str = None,
                    provided_aadhaar: str = None,
                    provided_pincode: str = None) -> dict:

    if provided_name is None:
        return {"verified": False, "reason": "name_missing"}

    if provided_name.strip() != account.full_name.strip():
        return {"verified": False, "reason": "name_mismatch"}

    if provided_dob is not None:
        if provided_dob.strip() == account.dob.strip():
            return {"verified": True, "reason": "name_and_dob"}

    if provided_aadhaar is not None:
        if provided_aadhaar.strip() == account.aadhaar_last4.strip():
            return {"verified": True, "reason": "name_and_aadhaar"}

    if provided_pincode is not None:
        if provided_pincode.strip() == account.pincode.strip():
            return {"verified": True, "reason": "name_and_pincode"}

    if provided_dob is None and provided_aadhaar is None and provided_pincode is None:
        return {"verified": False, "reason": "secondary_missing"}

    return {"verified": False, "reason": "secondary_mismatch"}

Writing verifier.py


In [11]:
%%writefile extractor.py

import os
import json
from datetime import datetime
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])


def _ask_groq(prompt: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


def extract_account_id(text: str) -> str | None:
    prompt = f"""
Extract the account ID from this text. Account IDs start with ACC followed by digits (like ACC1001, ACC1002).
Ignore spaces, punctuation, case differences — normalize to uppercase with no spaces.
Text: "{text}"
Reply with ONLY the account ID (e.g. ACC1001) or null if none found.
No explanation, no punctuation, just the ID or null.
"""
    result = _ask_groq(prompt).strip()
    if result.lower() == "null" or result == "":
        return None
    return result.upper().replace(" ", "")


def extract_full_name(text: str) -> str | None:
    prompt = f"""
Extract the person's full name from this text.
People may say "my name is X", "I am X", "you can call me Y but my full name is X".
Always return the FULL name, not a nickname.

CRITICAL RULES:
- Return the name EXACTLY as the user wrote it. Do NOT fix capitalization or formatting.
- If the user says "my name is nithin jain" return "nithin jain" not "Nithin Jain"
- Return ONLY the name itself, never include words like "my name is", "I am", "it is", "its"
- Only return null if there is clearly no name at all in the text

Text: "{text}"
Reply with ONLY the name exactly as written by the user, or null if no name found.
No explanation, just the name or null.
"""
    result = _ask_groq(prompt).strip()
    if result.lower() == "null" or result == "":
        return None
    return result


def extract_dob(text: str) -> str | None:
    prompt = f"""
Extract the date of birth from this text and convert it to YYYY-MM-DD format.
Handle all formats: "14th May 1990", "May 14 1990", "14-05-1990", "DOB is May 14, 90" etc.
For 2-digit years assume 1900s (so 90 = 1990).
Validate the date is real (e.g. Feb 29 only valid on leap years).
Text: "{text}"
Reply with ONLY the date in YYYY-MM-DD format or null if invalid or not found.
No explanation, just the date or null.
"""
    result = _ask_groq(prompt).strip()
    if result.lower() == "null" or result == "":
        return None
    try:
        datetime.strptime(result, "%Y-%m-%d")
        return result
    except ValueError:
        return None


def extract_aadhaar_last4(text: str) -> str | None:
    prompt = f"""
Extract the last 4 digits of the Aadhaar number from this text.
People may say "last four of my Aadhaar is 4321", "Aadhaar ends with 9876".
Text: "{text}"
Reply with ONLY the 4 digits or null if not found.
No explanation, just 4 digits or null.
"""
    result = _ask_groq(prompt).strip()
    if result.lower() == "null" or result == "" or not result.isdigit() or len(result) != 4:
        return None
    return result


def extract_pincode(text: str) -> str | None:
    prompt = f"""
Extract the 6-digit Indian pincode from this text.
People may say "pincode is 400001" or "it's 4 0 0 0 0 1" (digit by digit).
Remove all spaces between digits.
Text: "{text}"
Reply with ONLY the 6 digits or null if not found.
No explanation, just 6 digits or null.
"""
    result = _ask_groq(prompt).strip().replace(" ", "")
    if result.lower() == "null" or result == "" or not result.isdigit() or len(result) != 6:
        return None
    return result


def extract_payment_amount(text: str, balance: float) -> float | None:
    prompt = f"""
Extract the payment amount from this text. Outstanding balance is {balance}.
- "pay 500" → 500.0
- "a thousand rupees" → 1000.0
- "clear the full amount" / "pay all" / "pay everything" → {balance}
- "500 for now" → 500.0
Text: "{text}"
Reply with ONLY the numeric amount or null if not found. No currency symbol.
No explanation, just the number or null.
"""
    result = _ask_groq(prompt).strip()
    if result.lower() == "null" or result == "":
        return None
    try:
        return round(float(result), 2)
    except:
        return None


def extract_card_details(text: str) -> dict:
    prompt = f"""
Extract card payment details from this text.
- Card numbers may have spaces: "4532 0151 1283 0366" → "4532015112830366"
- Expiry: "December 2027" or "12/27" → month=12, year=2027
- CVV spoken: "one two three" → "123"
Text: "{text}"
Reply with ONLY this JSON (null for anything not found):
{{
  "card_number": "digits only no spaces",
  "expiry_month": integer or null,
  "expiry_year": 4-digit integer or null,
  "cvv": "digits only" or null,
  "cardholder_name": "name" or null
}}
No explanation, just the JSON.
"""
    result = _ask_groq(prompt).strip()
    result = result.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(result)
    except:
        return {
            "card_number": None,
            "expiry_month": None,
            "expiry_year": None,
            "cvv": None,
            "cardholder_name": None
        }


def looks_like_name_attempt(text: str) -> bool:
    prompt = f"""
Does this text look like someone is trying to provide their name, even if it's wrong or incomplete?
Examples that ARE name attempts: "Wrong Person", "John", "my name is xyz", "it's kumar"
Examples that are NOT name attempts: "I don't know", "what?", "help me", "skip this"

Text: "{text}"
Reply with ONLY yes or no.
"""
    result = _ask_groq(prompt).strip().lower()
    return result == "yes"



Overwriting extractor.py


In [12]:
%%writefile agent.py

from state import ConversationState, Stage, CardData
from tools import lookup_account, process_payment
from verifier import verify_identity
from extractor import (
    extract_account_id, extract_full_name, extract_dob,
    extract_aadhaar_last4, extract_pincode,
    extract_payment_amount, extract_card_details,
    looks_like_name_attempt
)

RETRYABLE_ERRORS = {"invalid_card", "invalid_cvv", "invalid_expiry"}


class Agent:
    def __init__(self):
        self.state = ConversationState()

    def next(self, user_input: str) -> dict:
        user_input = user_input.strip()
        try:
            response = self._handle(user_input)
        except Exception as e:
            response = f"Sorry, something went wrong on our end. Please try again. (Error: {str(e)})"
        return {"message": response}

    def _handle(self, user_input: str) -> str:
        s = self.state

        if s.stage == Stage.DONE:
            return "This conversation has ended. Please start a new session if you need further assistance."

        if s.stage == Stage.GREET:
            account_id = extract_account_id(user_input)
            if account_id:
                s.account_id = account_id
                s.stage = Stage.LOOKUP
                lookup_response = self._do_lookup()
                return "Hello! Welcome to the payment collection service.\n\n" + lookup_response
            s.stage = Stage.LOOKUP
            return (
                "Hello! Welcome to the payment collection service. "
                "Could you please share your account ID to get started?"
            )

        if s.stage == Stage.LOOKUP:
            account_id = extract_account_id(user_input)
            if not account_id:
                return (
                    "I couldn't find an account ID in your message. "
                    "Could you please share your account ID? (e.g. ACC1001)"
                )
            s.account_id = account_id
            return self._do_lookup()

        if s.stage == Stage.VERIFY:
            return self._handle_verify(user_input)

        if s.stage == Stage.SHOW_BALANCE:
            return self._handle_amount(user_input)

        if s.stage == Stage.COLLECT_CARD:
            return self._handle_card(user_input)

        if s.stage == Stage.PROCESS_PAYMENT:
            return self._do_payment()

        return "I'm not sure how to handle that. Could you please try again?"

    def _do_lookup(self) -> str:
        s = self.state
        try:
            account = lookup_account(s.account_id)
        except Exception as e:
            s.stage = Stage.LOOKUP
            return (
                "We're having trouble reaching our servers. "
                "Please try again. Could you re-enter your account ID?"
            )

        if account is None:
            saved_id = s.account_id
            s.account_id = None
            return (
                f"We couldn't find an account with ID '{saved_id}'. "
                "Please double-check your account ID and try again."
            )

        s.account_data = account
        s.stage = Stage.VERIFY
        return (
            "Got it! I found your account. "
            "For security purposes, I need to verify your identity. "
            "Could you please share your full name as it appears on your account?"
        )

    def _handle_verify(self, user_input: str) -> str:
        s = self.state

        extracted_name = extract_full_name(user_input)
        extracted_dob = extract_dob(user_input)
        extracted_aadhaar = extract_aadhaar_last4(user_input)
        extracted_pincode = extract_pincode(user_input)

        if s.provided_name is None:

            if extracted_name is None:
                if looks_like_name_attempt(user_input):
                    s.verify_attempts += 1
                    remaining = s.MAX_VERIFY_ATTEMPTS - s.verify_attempts
                    if remaining <= 0:
                        s.stage = Stage.DONE
                        return (
                            "I'm sorry, but we've been unable to verify your identity "
                            "after multiple attempts. For security reasons, this session "
                            "has been closed. Please contact customer support for assistance."
                        )
                    return (
                        "I was not able to catch your full name clearly. "
                        "Could you please share your full name exactly as it appears on your account? "
                        f"({remaining} attempt(s) remaining)"
                    )
                else:
                    return (
                        "Could you please share your full name "
                        "as it appears on your account?"
                    )

            name_check = verify_identity(s.account_data, extracted_name)
            if name_check["reason"] == "name_mismatch":
                s.verify_attempts += 1
                remaining = s.MAX_VERIFY_ATTEMPTS - s.verify_attempts
                if remaining <= 0:
                    s.stage = Stage.DONE
                    return (
                        "I'm sorry, but we've been unable to verify your identity "
                        "after multiple attempts. For security reasons, this session "
                        "has been closed. Please contact customer support for assistance."
                    )
                return (
                    "The name you provided does not match our records. "
                    "Please provide your full name exactly as it appears on your account. "
                    f"({remaining} attempt(s) remaining)"
                )

            s.provided_name = extracted_name

        if extracted_dob and s.provided_dob is None:
            s.provided_dob = extracted_dob
        if extracted_aadhaar and s.provided_aadhaar is None:
            s.provided_aadhaar = extracted_aadhaar
        if extracted_pincode and s.provided_pincode is None:
            s.provided_pincode = extracted_pincode

        if s.provided_dob is None and s.provided_aadhaar is None and s.provided_pincode is None:
            return (
                "Thank you. To complete verification, could you please "
                "provide one of the following: your date of birth, "
                "last 4 digits of your Aadhaar, or your pincode?"
            )

        result = verify_identity(
            s.account_data,
            s.provided_name,
            provided_dob=s.provided_dob,
            provided_aadhaar=s.provided_aadhaar,
            provided_pincode=s.provided_pincode
        )

        if result["verified"]:
            s.is_verified = True
            s.stage = Stage.SHOW_BALANCE
            balance = s.account_data.balance
            if balance == 0:
                s.stage = Stage.DONE
                return (
                    "Identity verified. Your current outstanding balance is Rs. 0.00. "
                    "You have no pending dues at this time. Have a great day!"
                )
            return (
                f"Identity verified. Your outstanding balance is Rs. {balance:,.2f}. "
                "How much would you like to pay? You can pay the full amount or a partial amount."
            )

        s.secondary_attempts += 1
        remaining = s.MAX_SECONDARY_ATTEMPTS - s.secondary_attempts

        if remaining <= 0:
            s.stage = Stage.DONE
            return (
                "I'm sorry, but we've been unable to verify your identity "
                "after multiple attempts. For security reasons, this session "
                "has been closed. Please contact customer support for assistance."
            )

        s.provided_dob = None
        s.provided_aadhaar = None
        s.provided_pincode = None

        return (
            "The details you provided do not match our records. "
            "Please try again with your date of birth, "
            "last 4 digits of Aadhaar, or pincode. "
            f"({remaining} attempt(s) remaining)"
        )

    def _handle_amount(self, user_input: str) -> str:
        s = self.state
        balance = s.account_data.balance
        amount = extract_payment_amount(user_input, balance)

        if amount is None:
            return (
                f"I couldn't understand the amount. "
                f"Your outstanding balance is Rs. {balance:,.2f}. "
                "How much would you like to pay?"
            )
        if amount <= 0:
            return (
                "The payment amount must be greater than zero. "
                "How much would you like to pay?"
            )
        if amount > balance:
            return (
                f"The amount Rs. {amount:,.2f} exceeds your outstanding "
                f"balance of Rs. {balance:,.2f}. "
                "Please enter an amount equal to or less than your balance."
            )

        s.payment_amount = amount
        s.stage = Stage.COLLECT_CARD
        return (
            f"Got it, Rs. {amount:,.2f} it is.\n\n"
            "Please share your card details:\n"
            "- Card number\n"
            "- Expiry date (month and year)\n"
            "- CVV\n"
            "- Name on card (if different from account name)"
        )

    def _handle_card(self, user_input: str) -> str:
        s = self.state
        card = s.card_data
        extracted = extract_card_details(user_input)

        if extracted.get("card_number"):
            card.card_number = extracted["card_number"]
        if extracted.get("expiry_month"):
            card.expiry_month = extracted["expiry_month"]
        if extracted.get("expiry_year"):
            card.expiry_year = extracted["expiry_year"]
        if extracted.get("cvv"):
            card.cvv = extracted["cvv"]
        if extracted.get("cardholder_name"):
            card.cardholder_name = extracted["cardholder_name"]

        if not card.cardholder_name:
            card.cardholder_name = s.account_data.full_name

        missing = []
        if not card.card_number:
            missing.append("card number")
        if not card.expiry_month or not card.expiry_year:
            missing.append("expiry date (month and year)")
        if not card.cvv:
            missing.append("CVV")

        if missing:
            return (
                f"Thanks! I still need your {', '.join(missing)}. "
                "Could you please provide that?"
            )

        s.stage = Stage.PROCESS_PAYMENT
        return self._do_payment()

    def _do_payment(self) -> str:
        s = self.state
        card = s.card_data

        card_payload = {
            "cardholder_name": card.cardholder_name,
            "card_number": card.card_number,
            "cvv": card.cvv,
            "expiry_month": card.expiry_month,
            "expiry_year": card.expiry_year
        }

        try:
            result = process_payment(s.account_id, s.payment_amount, card_payload)
        except Exception as e:
            s.stage = Stage.COLLECT_CARD
            return (
                "We encountered an error processing your payment. "
                "Please try again. Could you re-enter your card details?"
            )

        if result["success"]:
            s.card_data = CardData()
            s.transaction_id = result["transaction_id"]
            s.stage = Stage.DONE
            return (
                f"Your payment of Rs. {s.payment_amount:,.2f} has been processed successfully.\n\n"
                f"Account: {s.account_id}\n"
                f"Amount: Rs. {s.payment_amount:,.2f}\n"
                f"Transaction ID: {result['transaction_id']}\n\n"
                "Please save your transaction ID for your records. "
                "Thank you for your payment."
            )

        error_code = result.get("error_code", "unknown_error")

        if error_code in RETRYABLE_ERRORS:
            s.stage = Stage.COLLECT_CARD
            if error_code == "invalid_card":
                s.card_data.card_number = None
                return (
                    "The card number you entered does not appear to be valid. "
                    "Please check and re-enter your card number."
                )
            elif error_code == "invalid_cvv":
                s.card_data.cvv = None
                return (
                    "The CVV you entered does not match. "
                    "Please check the 3-digit code on the back of your card "
                    "and try again."
                )
            elif error_code == "invalid_expiry":
                s.card_data.expiry_month = None
                s.card_data.expiry_year = None
                return (
                    "The expiry date you entered is invalid or the card has expired. "
                    "Please provide a valid expiry date."
                )

        if error_code == "insufficient_balance":
            s.stage = Stage.SHOW_BALANCE
            s.payment_amount = None
            return (
                f"The payment could not go through as the amount exceeds "
                f"your current balance of Rs. {s.account_data.balance:,.2f}. "
                "Please enter a lower amount to continue."
            )

        s.stage = Stage.DONE
        terminal_messages = {
            "invalid_amount": (
                "The payment amount is invalid. "
                "Please start a new session and try again."
            ),
        }
        return terminal_messages.get(
            error_code,
            f"We were unable to process your payment ({error_code}). "
            "Please contact customer support for assistance."
        )

Overwriting agent.py


In [13]:
%%writefile run.py

from agent import Agent


def main():
    print("Payment Collection Agent")
    print("Type 'quit' or 'exit' to end")
    print()

    agent = Agent()
    opening = agent.next("Hi")
    print(f"Agent: {opening['message']}")
    print()

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if not user_input:
            continue

        if user_input.lower() in ("quit", "exit"):
            print("Agent: Thank you. Goodbye!")
            break

        response = agent.next(user_input)
        print()
        print(f"Agent: {response['message']}")
        print()

        if agent.state.stage == "done":
            break


if __name__ == "__main__":
    main()

Overwriting run.py


In [14]:
%%writefile eval.py


import sys
for mod in ['agent', 'extractor', 'verifier', 'tools', 'state']:
    if mod in sys.modules:
        del sys.modules[mod]

from agent import Agent


def run_conversation(turns, test_name):
    agent = Agent()
    passed = 0
    failed = 0

    print(f"\n{test_name}")
    print(f"{'-' * len(test_name)}")

    for i, (user_input, expected_keyword) in enumerate(turns):
        response = agent.next(user_input)["message"]

        if expected_keyword is None:
            ok = True
        else:
            ok = expected_keyword.lower() in response.lower()

        if ok:
            passed += 1
        else:
            failed += 1
            print(f"  [turn {i+1}] FAIL — expected '{expected_keyword}' in response")
            print(f"    user: {user_input}")
            print(f"    agent: {response[:100]}{'...' if len(response) > 100 else ''}")

    total = passed + failed
    score = round((passed / total) * 100, 1) if total > 0 else 0
    print(f"  passed {passed}/{total}")
    return {"test": test_name, "passed": passed, "failed": failed, "score": score}


def test_happy_path():
    return run_conversation([
        ("Hi there",                                                   "account"),
        ("yeah my account is ACC1001 I think",                         "found"),
        ("my name is Nithin Jain",                                     "date of birth"),
        ("I was born on 14th May 1990",                                "verified"),
        ("just clear the full amount",                                  "card"),
        ("card is 4532 0151 1283 0366 expires December 2027 cvv 123", "successful"),
    ], "Happy path — full payment with messy inputs")


def test_happy_path_aadhaar():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1001",                                                     "found"),
        ("Nithin Jain",                                                 "date of birth"),
        ("last four of my aadhaar is 4321",                            "verified"),
        ("500",                                                         "card"),
        ("card is 4532 0151 1283 0366 expires December 2027 cvv 123", "successful"),
    ], "Happy path — verify with Aadhaar")


def test_happy_path_pincode():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1001",                                                     "found"),
        ("Nithin Jain",                                                 "date of birth"),
        ("my pincode is 400001",                                        "verified"),
        ("pay 500",                                                     "card"),
        ("4532 0151 1283 0366 12/2027 cvv 123",                        "successful"),
    ], "Happy path — verify with pincode")


def test_partial_payment():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1001",                                                     "found"),
        ("Nithin Jain",                                                 "date of birth"),
        ("1990-05-14",                                                  "verified"),
        ("can I pay 500 for now?",                                      "500"),
        ("card is 4532 0151 1283 0366 expires December 2027 cvv 123", "successful"),
    ], "Partial payment")


def test_verification_name_fail_then_pass():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1001",                                                     "found"),
        ("nithin jain",                                                 "does not match"),
        ("Nithin Jain",                                                 "date of birth"),
        ("4321",                                                        "verified"),
        ("pay full amount",                                             "card"),
        ("4532 0151 1283 0366 expires December 2027 cvv 123",          "successful"),
    ], "Verification — wrong name then correct")


def test_verification_exhausted():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1001",         "found"),
        ("Wrong Person",    "not able"),
        ("Another Wrong",   "not able"),
        ("Still Wrong",     "closed"),
    ], "Verification — all attempts exhausted")


def test_secondary_factor_fail_then_pass():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1001",                                                     "found"),
        ("Nithin Jain",                                                 "date of birth"),
        ("1990-01-01",                                                  "do not match"),
        ("my aadhaar is 4321",                                         "verified"),
        ("pay full",                                                    "card"),
        ("4532 0151 1283 0366 expires December 2027 cvv 123",          "successful"),
    ], "Verification — wrong secondary factor then correct")


def test_invalid_account():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC9999",         "couldn't find"),
        ("ACC1001",         "found"),
        ("Nithin Jain",     "date of birth"),
    ], "Invalid account ID")


def test_zero_balance():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1003",         "found"),
        ("Priya Agarwal",   "date of birth"),
        ("10th August 1992","0.00"),
    ], "Zero balance account")


def test_long_name():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1002",                                                     "found"),
        ("Rajarajeswari Balasubramaniam",                               "date of birth"),
        ("aadhaar last 4 is 9876",                                     "verified"),
        ("clear full amount",                                           "card"),
        ("4532 0151 1283 0366 expires December 2027 cvv 123",          "successful"),
    ], "Long Indian name (ACC1002)")


def test_amount_exceeds_balance():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1001",         "found"),
        ("Nithin Jain",     "date of birth"),
        ("1990-05-14",      "verified"),
        ("pay 99999",       "exceeds"),
        ("pay 500",         "card"),
        ("4532 0151 1283 0366 expires December 2027 cvv 123",          "successful"),
    ], "Amount exceeds balance")


def test_card_details_across_turns():
    return run_conversation([
        ("Hi",                                                          "account"),
        ("ACC1001",                                                     "found"),
        ("Nithin Jain",                                                 "date of birth"),
        ("1990-05-14",                                                  "verified"),
        ("pay 500",                                                     "card"),
        ("card number is 4532 0151 1283 0366",                         "still need"),
        ("expires December 2027 cvv 123",                              "successful"),
    ], "Card details provided across multiple turns")


def test_leap_year_dob():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1004",         "found"),
        ("Rahul Mehta",     "date of birth"),
        ("29th Feb 1988",   "verified"),
        ("pay 1000",        "card"),
        ("4532 0151 1283 0366 expires December 2027 cvv 123",          "successful"),
    ], "Edge case — leap year date of birth (ACC1004)")


def test_session_closed_after_done():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1003",         "found"),
        ("Priya Agarwal",   "date of birth"),
        ("1992-08-10",      "0.00"),
        ("hello again",     "ended"),
    ], "Session closed after conversation ends")


def test_payment_invalid_card():
    return run_conversation([
        ("Hi",                                                              "account"),
        ("ACC1001",                                                         "found"),
        ("Nithin Jain",                                                     "date of birth"),
        ("1990-05-14",                                                      "verified"),
        ("pay 500",                                                         "card"),
        ("card 1234567890123456 expires december 2027 cvv 123",            "valid"),
        ("4532 0151 1283 0366 expires december 2027 cvv 123",              "successful"),
    ], "Payment failure — invalid card then retry")


def test_payment_invalid_cvv():
    return run_conversation([
        ("Hi",                                                              "account"),
        ("ACC1001",                                                         "found"),
        ("Nithin Jain",                                                     "date of birth"),
        ("1990-05-14",                                                      "verified"),
        ("pay 500",                                                         "card"),
        ("card 4532 0151 1283 0366 expires december 2027 cvv 999",        "cvv"),
        ("cvv is 123",                                                      "successful"),
    ], "Payment failure — invalid CVV then retry")


def test_payment_invalid_expiry():
    return run_conversation([
        ("Hi",                                                              "account"),
        ("ACC1001",                                                         "found"),
        ("Nithin Jain",                                                     "date of birth"),
        ("1990-05-14",                                                      "verified"),
        ("pay 500",                                                         "card"),
        ("card 4532 0151 1283 0366 expires january 2020 cvv 123",         "expiry"),
        ("expiry december 2027",                                            "successful"),
    ], "Payment failure — expired card then retry")


def test_secondary_factor_exhausted():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1001",         "found"),
        ("Nithin Jain",     "date of birth"),
        ("1990-01-01",      "do not match"),
        ("9999",            "do not match"),
        ("999999",          "closed"),
    ], "Verification — secondary factor all attempts exhausted")


def test_leap_year_nearby_date():
    return run_conversation([
        ("Hi",              "account"),
        ("ACC1004",         "found"),
        ("Rahul Mehta",     "date of birth"),
        ("28th Feb 1988",   "do not match"),
        ("29th Feb 1988",   "verified"),
        ("pay 1000",        "card"),
        ("4532 0151 1283 0366 expires december 2027 cvv 123", "successful"),
    ], "Edge case — nearby leap year date fails, correct date passes")


def run_all():
    tests = [
        test_happy_path,
        test_happy_path_aadhaar,
        test_happy_path_pincode,
        test_partial_payment,
        test_verification_name_fail_then_pass,
        test_verification_exhausted,
        test_secondary_factor_fail_then_pass,
        test_invalid_account,
        test_zero_balance,
        test_long_name,
        test_amount_exceeds_balance,
        test_card_details_across_turns,
        test_leap_year_dob,
        test_session_closed_after_done,
        test_payment_invalid_card,
        test_payment_invalid_cvv,
        test_payment_invalid_expiry,
        test_secondary_factor_exhausted,
        test_leap_year_nearby_date,
    ]

    all_results = []
    for test_fn in tests:
        try:
            result = test_fn()
            all_results.append(result)
        except Exception as e:
            print(f"\n{test_fn.__name__} — crashed: {str(e)}")
            all_results.append({
                "test": test_fn.__name__,
                "passed": 0,
                "failed": 1,
                "score": 0
            })

    total_passed = sum(r["passed"] for r in all_results)
    total_checks = sum(r["passed"] + r["failed"] for r in all_results)
    overall = round((total_passed / total_checks) * 100, 1) if total_checks > 0 else 0

    print(f"\n\nSummary")
    print(f"-------")
    for r in all_results:
        status = "ok" if r["failed"] == 0 else "fail"
        print(f"  [{status}] {r['test']} — {r['passed']}/{r['passed']+r['failed']}")

    print(f"\n  overall: {total_passed}/{total_checks} checks passed ({overall}%)")


if __name__ == "__main__":
    run_all()

Writing eval.py


In [15]:
%%writefile requirements.txt

groq>=0.9.0
requests>=2.31.0
python-dotenv>=1.0.0

Writing requirements.txt


In [16]:
import sys
for mod in ['agent', 'extractor', 'verifier', 'tools', 'state']:
    if mod in sys.modules:
        del sys.modules[mod]

from agent import Agent
a = Agent()
print(a.next("Hi")["message"])
print(a.next("ACC1001")["message"])
print(a.next("Nithin Jain")["message"])
print(a.next("1990-05-14")["message"])
print(a.next("pay 500")["message"])
print(a.next("4532 0151 1283 0366 expires december 2027 cvv 123")["message"])

Hello! Welcome to the payment collection service. Could you please share your account ID to get started?
Got it! I found your account. For security purposes, I need to verify your identity. Could you please share your full name as it appears on your account?
Thank you. To complete verification, could you please provide one of the following: your date of birth, last 4 digits of your Aadhaar, or your pincode?
Identity verified. Your outstanding balance is Rs. 1,250.75. How much would you like to pay? You can pay the full amount or a partial amount.
Got it, Rs. 500.00 it is.

Please share your card details:
- Card number
- Expiry date (month and year)
- CVV
- Name on card (if different from account name)
Your payment of Rs. 500.00 has been processed successfully.

Account: ACC1001
Amount: Rs. 500.00
Transaction ID: txn_1778999406623_2feskmt

Please save your transaction ID for your records. Thank you for your payment.


In [17]:
import sys
for mod in ['agent', 'extractor', 'verifier', 'tools', 'state', 'eval']:
    if mod in sys.modules:
        del sys.modules[mod]

from eval import run_all
run_all()


Happy path — full payment with messy inputs
-------------------------------------------
  passed 6/6

Happy path — verify with Aadhaar
--------------------------------
  passed 6/6

Happy path — verify with pincode
--------------------------------
  passed 6/6

Partial payment
---------------
  passed 6/6

Verification — wrong name then correct
--------------------------------------
  passed 7/7

Verification — all attempts exhausted
-------------------------------------
  passed 5/5

Verification — wrong secondary factor then correct
--------------------------------------------------
  passed 7/7

Invalid account ID
------------------
  passed 4/4

Zero balance account
--------------------
  passed 4/4

Long Indian name (ACC1002)
--------------------------
  passed 6/6

Amount exceeds balance
----------------------
  passed 7/7

Card details provided across multiple turns
-------------------------------------------
  passed 7/7

Edge case — leap year date of birth (ACC1004)
---------

In [18]:
from agent import Agent

agent = Agent()
opening = agent.next("Hi")
print(f"Agent: {opening['message']}\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Agent: Thank you. Goodbye!")
        break
    response = agent.next(user_input)
    print(f"\nAgent: {response['message']}\n")
    if agent.state.stage == "done":
        break

Agent: Hello! Welcome to the payment collection service. Could you please share your account ID to get started?

You: my account number is ACC 1002 i think

Agent: Got it! I found your account. For security purposes, I need to verify your identity. Could you please share your full name as it appears on your account?

You: you can call me Raja but my full name is Rajarajeswari Balasubramaniam

Agent: Thank you. To complete verification, could you please provide one of the following: your date of birth, last 4 digits of your Aadhaar, or your pincode?

You: aadhaar last 4 is 9 8 7 6

Agent: Identity verified. Your outstanding balance is Rs. 540.00. How much would you like to pay? You can pay the full amount or a partial amount.

You: just clear whatever i owe

Agent: Got it, Rs. 540.00 it is.

Please share your card details:
- Card number
- Expiry date (month and year)
- CVV
- Name on card (if different from account name)

You: 4532 0151 1283 0366 expires december 2027 cvv one two three



In [20]:
from agent import Agent

agent = Agent()
opening = agent.next("Hi")
print(f"Agent: {opening['message']}\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Agent: Thank you. Goodbye!")
        break
    response = agent.next(user_input)
    print(f"\nAgent: {response['message']}\n")
    if agent.state.stage == "done":
        break

Agent: Hello! Welcome to the payment collection service. Could you please share your account ID to get started?

You: ACC1001

Agent: Got it! I found your account. For security purposes, I need to verify your identity. Could you please share your full name as it appears on your account?

You: Nithin Jain

Agent: Thank you. To complete verification, could you please provide one of the following: your date of birth, last 4 digits of your Aadhaar, or your pincode?

You: 4321

Agent: Identity verified. Your outstanding balance is Rs. 1,250.75. How much would you like to pay? You can pay the full amount or a partial amount.

You: 500

Agent: Got it, Rs. 500.00 it is.

Please share your card details:
- Card number
- Expiry date (month and year)
- CVV
- Name on card (if different from account name)

You: card number 4532 0151 1283 0366 

Agent: Thanks! I still need your expiry date (month and year), CVV. Could you please provide that?

You: expires december 2027

Agent: Thanks! I still need y

In [21]:
%%writefile README.md

# Payment Collection AI Agent

A conversational AI agent that handles end-to-end payment collection flows. The agent guides users through account lookup, identity verification, and card payment processing via natural conversation.

## Project Structure

    payment_agent/
    ├── agent.py          # Main Agent class — entry point for all conversations
    ├── state.py          # Conversation state definitions and stage constants
    ├── tools.py          # API calls — account lookup and payment processing
    ├── verifier.py       # Identity verification logic
    ├── extractor.py      # NLP extraction using Groq (Llama 3.3)
    ├── run.py            # Interactive CLI to chat with the agent
    ├── eval.py           # Automated evaluation with 14 test cases
    └── requirements.txt  # Python dependencies

## Setup

### 1. Clone the repository

    git clone <your-repo-url>
    cd payment_agent

### 2. Install dependencies

    pip install -r requirements.txt

### 3. Set your Groq API key

Create a .env file in the project root:

    GROQ_API_KEY=your-groq-api-key-here

Get a free API key at: https://console.groq.com

### 4. Run the agent interactively

    python run.py

### 5. Run the evaluation suite

    python eval.py

Note: The eval suite runs 14 tests sequentially. On Groq free tier you may hit rate limits. If this happens wait a few minutes and run again.

## How It Works

The agent follows a strict 8-step flow:

1. Greet the user and ask for account ID
2. Look up the account via API
3. Collect and verify identity — name plus one secondary factor
4. Show outstanding balance
5. Collect payment amount
6. Collect card details
7. Process payment via API
8. Confirm outcome with full payment summary

## Identity Verification

Verification requires full name exact match (case sensitive) plus at least one of: date of birth, last 4 digits of Aadhaar, or pincode.

Users get 3 attempts for name verification and 3 attempts for secondary factor verification before the session is closed for security.

Sensitive data (DOB, Aadhaar, pincode) is never shown back to the user at any point.

## Handling Messy Input

The agent uses Groq Llama 3.3-70b to extract structured data from natural language:

    User says                                           Agent understands
    "yeah my account is ACC 1001 i think"              ACC1001
    "you can call me Raja but full name is             Rajarajeswari Balasubramaniam
     Rajarajeswari Balasubramaniam"
    "I was born 14th may 1990"                         1990-05-14
    "card 4532 0151 1283 0366 expires dec 2027         full card details extracted
     cvv one two three"

## Test Accounts

    Account ID   Name                              Balance
    ACC1001      Nithin Jain                       1250.75
    ACC1002      Rajarajeswari Balasubramaniam     540.00
    ACC1003      Priya Agarwal                     0.00
    ACC1004      Rahul Mehta (leap year DOB)       3200.50

## Sample Conversations

See sample_conversations.md for full conversation examples covering success, failure, and edge cases.

## Evaluation

The eval suite covers 14 test cases including happy paths, verification failures, payment failures, and edge cases.

Run with:

    python eval.py

Result: 82/82 checks passed (100%)

Note on rate limits: Groq free tier has per-minute token limits. If the full eval hits rate limits, wait a few minutes and rerun.

Writing README.md


In [22]:
%%writefile DESIGN.md

# Design Document — Payment Collection AI Agent

## Architecture Overview

The agent is structured as a set of single-responsibility modules that the main Agent class orchestrates:

    User Input
        |
        v
    agent.py (Agent.next)
        |
        |-- extractor.py   Extract structured data from messy natural language
        |-- verifier.py    Strict identity verification logic
        |-- tools.py       API calls to lookup and payment endpoints
        |-- state.py       Full conversation state held in memory
        |
        v
    Agent Response

Each call to Agent.next() represents one turn. The agent reads the current stage from ConversationState, processes the input, updates state, and returns a response. No external memory or database is used — all state lives in the ConversationState object for the duration of the session.

## Module Breakdown

### agent.py
The central orchestrator. Contains the Agent class with a single public method next(user_input) that returns a dict with a message key. Internally routes to stage-specific handlers: _handle_verify, _handle_amount, _handle_card, _do_payment. Each handler reads and mutates ConversationState.

### state.py
Defines three dataclasses: AccountData (API response), CardData (card fields collected across turns), and ConversationState (full session memory). Stage constants are defined as a class with string values. Separating state into a dedicated module prevents it from being scattered across business logic.

### tools.py
Two functions: lookup_account and process_payment. Each makes one HTTP POST request to the provided API, handles all status codes, and raises clear exceptions on network failures. No business logic lives here — just clean API calls.

### verifier.py
Pure Python verification logic with no LLM dependency. Implements the exact rules from the spec: name must match exactly (case sensitive, strip whitespace only), plus at least one secondary factor (DOB, Aadhaar last 4, or pincode) must match. Returns a dict with verified and reason fields so the caller can respond precisely to each failure mode.

### extractor.py
Uses Groq (Llama 3.3-70b) to extract structured data from free-form user input. Each turn makes one API call extracting all possible fields at once. The LLM returns structured text which is then validated in Python (date parsing with datetime.strptime, digit length checks). This is the only module with an LLM dependency.

### run.py
A simple CLI loop for interactive testing. Calls Agent.next() in a loop and prints responses. Stops when the agent reaches Stage.DONE.

### eval.py
14 automated test cases covering happy paths, verification failures, payment failures, and edge cases. Each test runs a full conversation and checks that each agent response contains an expected keyword. Reports per-test and overall pass rates.

## Key Design Decisions

### LLM for extraction, Python for logic
The LLM is used only for one thing: extracting structured data from messy natural language. All business logic — verification, stage transitions, retry counting, payment validation — is deterministic Python. This makes the agent predictable and easy to test. If the LLM extracts the wrong thing, the Python logic catches it with validation.

### Stage machine over open-ended conversation
The agent uses a strict stage machine (GREET, LOOKUP, VERIFY, SHOW_BALANCE, COLLECT_CARD, PROCESS_PAYMENT, DONE) rather than a free-form LLM conversation. This guarantees the flow is always followed in order, sensitive data is never skipped over, and verification always happens before payment. It also makes the agent fully deterministic and testable.

### Separate retry counters for name and secondary factor
An early version used a single verify_attempts counter for both name mismatches and secondary factor mismatches. This was unfair to users. The final version uses separate counters (verify_attempts for name, secondary_attempts for secondary factor), giving users a fair number of attempts at each step independently.

### Strict name matching
The spec requires no fuzzy matching. The extractor is explicitly prompted to return the name exactly as the user typed it, preserving case. The verifier then does an exact string comparison after stripping whitespace only. This means nithin jain correctly fails against Nithin Jain.

### Card data cleared after payment
Raw card data is cleared from ConversationState immediately after the payment API call regardless of success or failure. This minimises the time sensitive card data lives in memory.

### Retryable vs terminal payment errors
Not all payment failures are equal. Invalid card number, CVV, and expiry are user-fixable — the agent resets only the failing field and asks the user to try again. Insufficient balance routes back to the amount collection step. Unknown errors close the session cleanly.

### Python-side date validation
After the LLM extracts a date string, Python validates it with datetime.strptime before accepting it. This catches cases where the LLM returns an invalid date like Feb 29 on a non-leap year, which would otherwise silently fail verification.

## Tradeoffs

### Groq free tier rate limits
The free tier has per-minute token limits that make running the full eval suite in one pass occasionally difficult. This is a deployment constraint not a code issue. In production this would be solved by using a paid API tier or a self-hosted model.

### LLM non-determinism
Groq with temperature=0 is highly consistent but not guaranteed to be identical across runs. Edge cases like ambiguous names or unusual date formats may occasionally be extracted incorrectly. The Python-side validation catches the most critical errors.

### No conversation history sent to LLM
Each extraction prompt contains only the current user message, not the full conversation history. This keeps prompts short and fast but means the LLM cannot use context from earlier turns. The ConversationState handles context instead — once a field is stored it is never re-asked.

### Session is not resumable
ConversationState lives in memory for the duration of one Agent instance. If the process restarts, the session is lost. In production this would be stored in a database keyed by session ID.

## What I Would Improve With More Time

1. Persistent session storage so conversations survive process restarts
2. A paid LLM tier to eliminate rate limit issues in automated eval
3. Confidence scoring on LLM extractions — if the model is uncertain, ask the user to clarify
4. Webhook or async payment processing for slow API responses
5. Structured logging of all turns for debugging and audit
6. A web interface instead of CLI for easier demonstration
7. Multi-language support for regional Indian languages

Writing DESIGN.md


In [23]:
%%writefile sample_conversations.md

# Sample Conversations

## 1. Successful End-to-End Payment

Agent: Hello! Welcome to the payment collection service. Could you please share your account ID to get started?

You: yeah my account is ACC1001 i think

Agent: Got it! I found your account. Could you please share your full name?

You: its Nithin Jain

Agent: Thank you. Could you verify with your date of birth, Aadhaar last 4, or pincode?

You: my aadhaar ends with 4321

Agent: Identity verified. Your outstanding balance is Rs. 1,250.75. How much would you like to pay?

You: pay everything

Agent: Got it, Rs. 1,250.75 it is. Please share your card details.

You: 4532 0151 1283 0366 expires december 2027 cvv 123

Agent: Your payment of Rs. 1,250.75 has been processed successfully.
Account: ACC1001
Amount: Rs. 1,250.75
Transaction ID: txn_1778912678543_urp0e9i
Please save your transaction ID for your records. Thank you for your payment.

---

## 2. Verification Failure — All Retries Exhausted

Agent: Could you please share your full name as it appears on your account?

You: wrong name

Agent: The name you provided does not match our records. Please provide your full name exactly as it appears on your account. (2 attempt(s) remaining)

You: another wrong name

Agent: The name you provided does not match our records. Please provide your full name exactly as it appears on your account. (1 attempt(s) remaining)

You: still wrong

Agent: I'm sorry, but we've been unable to verify your identity after multiple attempts. For security reasons, this session has been closed. Please contact customer support for assistance.

---

## 3. Payment Failure — Invalid Card with Retry

Agent: Please share your card details.

You: card 1234567890123456 expires december 2027 cvv 123

Agent: The card number you entered does not appear to be valid. Please check and try again.

You: 4532 0151 1283 0366 expires december 2027 cvv 123

Agent: Your payment of Rs. 500.00 has been processed successfully.
Transaction ID: txn_1778910789049_gulpfzp

---

## 4. Edge Case — Messy Input Throughout

Agent: Hello! Welcome to the payment collection service. Could you please share your account ID to get started?

You: my account number is ACC 1002 i think

Agent: Got it! I found your account. Could you please share your full name?

You: Rajarajeswari Balasubramaniam

Agent: Thank you. Could you verify with your date of birth, Aadhaar last 4, or pincode?

You: aadhaar last 4 is 9876

Agent: Identity verified. Your outstanding balance is Rs. 540.00. How much would you like to pay?

You: just clear whatever i owe

Agent: Got it, Rs. 540.00 it is. Please share your card details.

You: 4532 0151 1283 0366 expires december 2027 cvv one two three

Agent: Your payment of Rs. 540.00 has been processed successfully.
Transaction ID: txn_1778947289517_nnsm89h

---

## 5. Edge Case — Zero Balance Account

Agent: Could you please share your full name?

You: Priya Agarwal

Agent: Could you verify with your date of birth, Aadhaar last 4, or pincode?

You: 10th August 1992

Agent: Identity verified. Your current outstanding balance is Rs. 0.00. You have no pending dues at this time. Have a great day!

Writing sample_conversations.md
